# Load Modeling

CIM offers multiple means of modeling loads in various levels of detail using the classes EnergyConsumer, ConformLoad, and NonConformLoad. Figure 21 below illustrates four levels of modeling accuracy, which will be described in detail below.

## Distribution Load Modeling

Customer loads can be represented at three levels of detail depending on the requirements of the modeling effort. Each load in the distribution feeder is defined as an EnergyConsumer object, which may be a single metered customer or composed of multiple customers, as specified by the customerCount attribute. Like all ConductingEquipment, every EnergyConsumer has an associated Terminal to which it is connected. 

The simplest method places an aggregated EnergyConsumer object at the terminal where the high-side of the service transformer would be connected, as depicted in Figure 21 a). A single Terminal object from the EnergyConsumer is associated with the ConnectivityNode to which it is connected. The first two representations can be used for both single-phase and three-phase loads.

The second level of detail includes the customer transformer as a simple two-winding PowerTransformer object with a single Terminal associated with the EnergyConsumer object and the ConnectivityNode to which the transformer low-side winding is connected, as depicted in Fig 21 b). Multiple customers served by the service transformer can be aggregated into a single EnergyConsumer object representing the total load. 

The last and most detailed representation explicitly models the split-phase secondary transformer, two-phase triplex lines, and individual unbalanced single-phase customer loads, as shown in Figure 21 c). Multiple customers are often connected to the low-side of the transformer, each with a length of overhead or underground secondary triplex line. Due to the split-phase configuration, the individual s1 and s2 phases of the EnergyConsumer and triplex ACLineSegment are considered a single object with each modeled with just one Terminal for both phases. However, the split-phase TransformerTank is topologically represented by three Terminal and two ConnectivityNode objects because the s1 and s2 phases are connected to different windings of the service transformer. 

Phasing of loads is specified by the EnergyConsumerPhase:phase attribute. For three-phase delta loads, the EnergyConsumer:phaseConnection is D and the three reverse-associated EnergyConsumerPhase instances will have phase=A for the AB load, phase=B for the BC load and phase=C for the AC load. A three-phase wye load may have either Y or Yn for the phaseConnection. 

Load values can be defined in two ways. The first is by directly specifying the real and reactive power values of the load through the EnergyConsumer:p and EnergyConsumer:q attributes. The second is as a ZIP load with specified constant impedance (Z), constant current (I), and constant power (P) components specified by the attributes of the LoadResponseCharacteristic class. 

Split-phase secondary loads use phase=s1 and phase=s2 to represent the unbalanced load components of household loads in North America. Plug loads and lighting are connected at 120V from s1 or s2 phase to neutral and appear as unbalanced load components specified by the EnergyConsumerPhase:p and EnergyConsumerPhase:q attributes for each s1 or s2 phase load object. Larger loads are connected at 240V from s1 to s2 are divided equally into the phase load p and q values. If the secondary triplex lines and split-phase transformer are explicitly modeled, phaseConnection=Y.

In [1]:
from cimgraph import utils
from mermaid import Mermaid
import cimgraph.data_profile.cim17v40 as cim

In [10]:
diagram_text = utils.get_mermaid([cim.Substation,cim.Feeder, cim.Terminal, cim.Equipment, cim.EquipmentContainer, cim.ConductingEquipment, cim.PowerSystemResource, cim.PowerCutZone, cim.EnergyConsumer, cim.EnergyConnection, cim.EnergyConsumerPhase, cim.LoadResponseCharacteristic, cim.PhaseShuntConnectionKind, cim.SinglePhaseKind])
Mermaid(diagram_text)

## Transmission Load Modeling

The EnergyConsumer class can also be used to represent transmission loads at the substation level. All transmission loads are assumed to be three-phase balanced, and so EnergyConsumerPhase is not specified. Loads are categorized as ConformLoad and NonConformLoad depending on whether the load follows a daily pattern. ConformLoad objects are the most common representation of aggregated feeder load at the substation level. NonConformLoad objects are sometimes used in studies and simulations to represent load shedding (with a negative value) or cold load pickup values after a blackout. All EnergyConsumer objects that follow the same load curve can be aggregated into a ConformLoadGroup or NonConformLoadGroup, as shown in Figure 23.

In [22]:
diagram_text = utils.get_mermaid([cim.EnergyArea, cim.LoadArea, cim.SubLoadArea, cim.LoadGroup, cim.ConformLoadGroup, cim.NonConformLoadGroup, cim.ConformLoadSchedule, cim.SeasonDayTypeSchedule, cim.Season, cim.DayType, cim.RegularIntervalSchedule, cim.NonConformLoadSchedule, cim.ConformLoad, cim.NonConformLoad, cim.StationSupply, cim.EnergyConsumer, cim.PowerCutZone, cim.Equipment, cim.ConductingEquipment, cim.LoadResponseCharacteristic, cim.EquipmentContainer, cim.Substation])
Mermaid(diagram_text)

Real and reactive load curves for all loads within a particular group are specified using either ConformLoadSchedule or NonConformLoadSchedule, which both inherit from the SeasonDayTypeSchedule class. This is also the parent class for schedules of Switch, TapChanger, and RegulatingControl objects, as shown in Figure 24 below. Each schedule is associated with a DayType and a Season, which are used to group similar days, such as weekdays, weekends, and holidays. 

The load curve itself is defined as a series of RegularTimePoint objects, where the time between consecutive points is equal. Following the practice of many datastores to omit repeated values, the set of time points do not need to be sequential. The timestamp associated with a particular point is not specified as a date/time, but rather using the sequenceNumber attribute. The timestamp is calculated by multiplying the value of the sequenceNumber attribute by the time interval between points in the load curve. The starting time has a sequenceNumber of zero and is specified by the startTime attribute of the BasicIntervalSchedule class.

In [32]:
diagram_text = utils.get_mermaid([cim.RegularIntervalSchedule, cim.RegularTimePoint, cim.SeasonDayTypeSchedule, cim.BasicIntervalSchedule, cim.IdentifiedObject, cim.Season, cim.DayType, cim.RegulationSchedule, cim.RegulatingControl, cim.TapSchedule, cim.TapChanger, cim.SwitchSchedule, cim.Switch, cim.ConformLoadSchedule, cim.NonConformLoadSchedule])
Mermaid(diagram_text)